# Quantum Tanner codes

Quantum Tanner codes are CSS codes built from a finite group, two sets of its members, and small classical codes called local codes.
Leverrier and Zémor introduced them as a family of asymptotically good quantum LDPC codes ([arXiv:2202.13641](https://arxiv.org/abs/2202.13641)).
Small instances are also candidates for near-term hardware: for example, Leverrier, Rozendaal, and Zémor report a $[[144, 12, 11]]$ code whose checks each act on 9 qubits ([arXiv:2512.20532](https://arxiv.org/abs/2512.20532)).

This notebook shows how to build quantum Tanner codes in qLDPC, and how to read off the properties that decide whether one suits a device: its parameters $[[n, k, d]]$, the weight of its checks, and the number of checks that act on each qubit.
It builds the toric code and a small code over a nonabelian group as `QTCode`s, inspects their local checks and symmetries, and rebuilds the published $[[144, 12, 11]]$ code as an `LQTCode`.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import time

import numpy as np

from qldpc import abstract, codes
from qldpc.objects import Pauli

## How a quantum Tanner code is built

A quantum Tanner code starts from a group $G$ and two sets $A$ and $B$ of its members.
Its square complex has a face $\{g, ag, gb, agb\}$ for each $g \in G$, $a \in A$, and $b \in B$, and several of these triples $(g, a, b)$ can describe the same face.
Each vertex of the complex is a corner of $|A| \times |B|$ faces, one for each pair $(a, b)$, which form the *local view* of the vertex.
Half of the vertices carry X-type checks, and the other half Z-type checks.
The X-type checks of a vertex span the tensor product $C_A \otimes C_B$ of two classical local codes, read on the $|A| \times |B|$ grid of its local view, and the Z-type checks of a vertex span the product of their dual codes.
The local views of two vertices with checks of different types meet on whole rows or columns of their grids, where words of a local code meet words of its dual, so the checks commute.

The *weight* of a check is the number of qubits that it acts on, and the *degree* of a qubit is the number of checks that act on it.
Both matter for measuring the checks on hardware.

qLDPC builds these codes with two classes:

- `QTCode` takes two sets $A$ and $B$ of distinct members, each of which holds the inverse of each of its members, and two local codes.
  It puts one qubit on each distinct face.
- `LQTCode` takes an explicit group, ordered lists of members that may repeat, and four local codes, as in [arXiv:2512.20532](https://arxiv.org/abs/2512.20532).
  It puts one qubit on each *square* $(i, j, g)$, for each position $i$ in $A$, position $j$ in $B$, and member $g$ of $G$, so it has $|A| \times |B| \times |G|$ qubits.

## The toric code

In the group $\mathbb{Z}_4 \times \mathbb{Z}_4$ with generators $x$ and $y$, take $A = \{x, x^{-1}\}$ and $B = \{y, y^{-1}\}$, and the repetition code $[2, 1, 2]$ as both local codes.
The $64$ triples $(g, a, b)$ describe $16$ distinct faces, each by four triples, so the code has $16$ qubits.
Each local check space holds one nonzero word, which acts on all four faces at a vertex, so the checks are those of a rotated toric code.

By default, `QTCode` builds a minimum-weight basis of each local check space (`local_checks="min_weight"`).

In [3]:
torus = abstract.Group.product(abstract.CyclicGroup(4), repeat=2)
shift_x, shift_y = torus.generators
toric_code = codes.QTCode([shift_x, ~shift_x], [shift_y, ~shift_y], codes.RepetitionCode(2))

assert toric_code.get_code_params() == (16, 2, 4)
print("[[n, k, d]]:", toric_code.get_code_params())
print("check weight:", toric_code.get_weight())

[[n, k, d]]: (16, 2, 4)
check weight: 4


## A code over a nonabelian group

For an example with more logical qubits, take larger local codes over a nonabelian group.
In the symmetric group $S_3$, take $A$ to be the three transpositions, and $B$ to be a rotation $r$, its inverse, and one transposition.
The local codes are the $[3, 1, 3]$ repetition code $C_A$ and its dual, the $[3, 2, 2]$ code of words with even weight, which `QTCode` uses for $C_B$ by default.
qLDPC represents group members as permutations, here of three points.

In [4]:
transpositions = [abstract.GroupMember(image) for image in ([1, 0, 2], [2, 1, 0], [0, 2, 1])]
rotation = abstract.GroupMember([1, 2, 0])
code = codes.QTCode(
    transpositions, [rotation, ~rotation, transpositions[0]], codes.RepetitionCode(3)
)

print("qubits:", len(code))
print("logical qubits:", code.dimension)
print("check weight:", code.get_weight())

qubits: 54
logical qubits: 11
check weight: 6


## Local checks

`get_local_views` returns the qubits at the local coordinates of every vertex with checks of one type.
Coordinate $i$ on the $A$ side belongs to `code.members_a[i]`, which lists the members of $A$ in increasing order, and likewise for $B$.
The checks of each vertex are consecutive rows of `code.matrix_x` or `code.matrix_z`, so reading them on the local view of their vertex shows their product structure.
The rows of these grids are the coordinates on the $A$ side, and their columns those on the $B$ side.

In [5]:
local_views = code.get_local_views(Pauli.X)
print("vertices with X-type checks, |A|, |B|:", local_views.shape)
local_view = local_views[0]
print("qubits at the local coordinates of the first of these vertices:")
print(local_view)

num_checks = len(code.matrix_x) // len(local_views)
for check in code.matrix_x[:num_checks]:
    print("check at this vertex:")
    print(check[local_view].view(np.ndarray))

vertices with X-type checks, |A|, |B|: (12, 3, 3)
qubits at the local coordinates of the first of these vertices:
[[0 1 2]
 [3 4 5]
 [6 7 8]]
check at this vertex:
[[1 0 1]
 [1 0 1]
 [1 0 1]]
check at this vertex:
[[0 1 1]
 [0 1 1]
 [0 1 1]]


Each check is the product of the repetition-code word $(1, 1, 1)$ on the $A$ side with a word of weight 2 of the even-weight code on the $B$ side, so it acts on $3 \times 2 = 6$ qubits.
These checks form a minimum-weight basis of the local check space: no basis of the space has lighter words.
The Z-type checks are products of words of the dual codes, and also have weight $2 \times 3 = 6$.

Minimum-weight checks are not chosen to make the degree of qubits small.
The section on the lifted $[[144, 12, 11]]$ code below shows how to choose the checks of each vertex instead.

In [6]:
def get_max_degree(code: codes.CSSCode) -> int:
    """The largest number of checks that act on one qubit."""
    checks = np.vstack([code.matrix_x, code.matrix_z]).view(np.ndarray)
    return int(np.count_nonzero(checks, axis=0).max())


print("largest qubit degree:", get_max_degree(code))

largest qubit degree: 7


## Symmetries and an exact distance

`get_known_qudit_automorphism_group` returns a group of qubit permutations that preserve the code.
It contains translations of the complex, which multiply its vertices by members of the group.
With the optional `pybliss` package (`pip install 'qldpc[bliss]'`), it also contains the permutations that the graph-automorphism tool bliss finds among the parity checks, and among the light words of the local check spaces.
The group can miss other symmetries, so its order is a lower bound on the order of the full automorphism group of the code.
For this code, the group has order 72 with `pybliss`, and order 24, that of the translations alone, without it.
The outputs saved in this notebook were computed with `pybliss` installed.
Exact distance searches can use these symmetries to skip candidates that are equivalent to ones already searched.
Their heuristics decide whether a search asks for them, and the search below does not.

In [7]:
print("known symmetries:", code.get_known_qudit_automorphism_group().order)

distance = code.get_distance_exact()
assert (len(code), code.dimension, distance) == (54, 11, 4)
print("[[n, k, d]]:", (len(code), code.dimension, distance))

known symmetries: 72


[[n, k, d]]: (54, 11, 4)


The search is exact, so this is a $[[54, 11, 4]]$ code with checks of weight 6.
For comparison, eleven copies of the $[[16, 1, 4]]$ rotated surface code use 176 qubits, with checks of weight at most 4 that act on neighboring qubits in a plane.
Unlike those surface-code blocks, this $S_3$ construction does not come with a planar nearest-neighbor layout.

## A lifted code: the published $[[144, 12, 11]]$ code

Leverrier, Rozendaal, and Zémor (LRZ) write quantum Tanner codes as lifts, in their Eq. (2) ([arXiv:2512.20532](https://arxiv.org/abs/2512.20532)).
Their data are a group $G$, ordered lists $A$ and $B$ of its members, which may repeat, and four binary codes $C_0$, $C_1$, $C'_0$, and $C'_1$.
Each code comes with a parity check matrix and a generator matrix, which LRZ write as $H_s$ and $G_s$ for $C_s$; the subscript distinguishes a generator matrix $G_s$ from the group $G$.
`LQTCode.from_lrz` takes these data in their notation, and converts them to the conventions of `LQTCode`, which treat $A$ and $B$ alike (see `help(codes.LQTCode.from_lrz)`).

The code below is the $[[144, 12, 11]]$ code of their Table 1(b), over $C_2 \times C_2$ with generators $x$ and $y$.
$C_0 = C'_0$ is the shortened Hamming code $[6, 3, 3]$, with the parity check and generator matrices of their Section 2, and $C_1 = C'_1$ is $C_0$ with its coordinates shifted cyclically by one.
We read the lists $A$ and $B$, and the code $C_1$, off the check matrices that LRZ publish with their paper.
`ClassicalCode.set_generator` gives each code its generator matrix.

In [8]:
group = abstract.Group.product(abstract.CyclicGroup(2), repeat=2)
shift_x, shift_y = group.generators
identity = group.identity
members_a = [identity, identity, identity, shift_x, shift_y, shift_x * shift_y]
members_b = [identity, identity, shift_x, shift_x, shift_y, shift_y]

matrix = np.array([[1, 0, 0, 0, 1, 1], [0, 1, 0, 1, 0, 1], [0, 0, 1, 1, 1, 0]])
generator = np.array([[0, 1, 1, 1, 0, 0], [1, 0, 1, 0, 1, 0], [1, 1, 0, 0, 0, 1]])
code_0 = codes.ClassicalCode(matrix).set_generator(generator)
code_1 = codes.ClassicalCode(np.roll(matrix, -1, axis=1))
code_1.set_generator(np.roll(generator, -1, axis=1))

lrz_codes = {
    local_checks: codes.LQTCode.from_lrz(
        group,
        members_a,
        members_b,
        code_0=code_0,
        code_1=code_1,
        code_0_prime=code_0,
        code_1_prime=code_1,
        local_checks=local_checks,
    )
    for local_checks in ["min_weight", "from_codes"]
}
for local_checks, lrz_code in lrz_codes.items():
    assert (len(lrz_code), lrz_code.dimension) == (144, 12)
    print(
        f"{local_checks}: [[n, k]] = [[{len(lrz_code)}, {lrz_code.dimension}]],"
        f" check weight {lrz_code.get_weight()}, largest qubit degree {get_max_degree(lrz_code)}"
    )

min_weight: [[n, k]] = [[144, 12]], check weight 9, largest qubit degree 16
from_codes: [[n, k]] = [[144, 12]], check weight 9, largest qubit degree 10


Both choices of local checks give the same stabilizer group, with checks of weight 9.
They differ in the degree of qubits.
`"from_codes"` uses the products of the rows of the matrices that the local codes carry, which are the checks that LRZ publish, up to the order of rows and columns; qLDPC's tests compare them with LRZ's files.
The degree of qubits limits how shallow a circuit that measures the checks can be.
To control it, build each local code from the parity check matrix that you want, and give it the generator matrix that you want with `ClassicalCode.set_generator`.
Then pass `local_checks="from_codes"`, which uses both matrices.

### Optional: the distance of the lifted code

LRZ report $d = 11$ from a randomized search, which gives an upper bound on the distance.
The exact search in the next cell finds $d = 11$.
It takes seconds once qLDPC has compiled its search functions, which can take longer the first time.
Set `compute_distance = True` to run it.

In [9]:
compute_distance = False

if compute_distance:
    start = time.perf_counter()
    lrz_distance = lrz_codes["from_codes"].get_distance_exact()
    print(f"distance: {lrz_distance}, found in {time.perf_counter() - start:.1f} seconds")

## Where to go next

- The [API reference of `qldpc.codes.quantum.quantum_tanner`](https://qldpc.readthedocs.io/autoapi/qldpc/codes/quantum/quantum_tanner/index.html) describes the choices of local checks, the conventions of `QTCode` and `LQTCode`, and how the stabilizer groups of `QTCode`s relate to those of `LQTCode`s.
  `QTCode.from_ordered_members` builds a `QTCode` whose local codes label their coordinates in a given order of members.
- Leverrier and Zémor introduced quantum Tanner codes in [arXiv:2202.13641](https://arxiv.org/abs/2202.13641), and their quadripartite complex in [arXiv:2206.07571](https://arxiv.org/abs/2206.07571).
- [arXiv:2508.05095](https://arxiv.org/abs/2508.05095) and [arXiv:2512.20532](https://arxiv.org/abs/2512.20532) construct explicit small instances.